# 第三问 Baseline：固定时刻更新计划的实际费用对比

本 notebook 对比两种固定更新策略：

- **策略一：06:00、12:00 各更新一次**。00:00 先制定全天计划，06:00 只调整 `[06:00, 12:00)`，12:00 只调整 `[12:00, 18:00)`。
- **策略二：仅 12:00 更新一次**。00:00 先制定全天计划，12:00 只调整 `[12:00, 18:00)`。

两种策略都不调整已经执行的时段，也不调整当前 6 小时窗口之外的更远时段。费用按第三问主口径计算：

$$F=F_{plan}+F_{adjust}+F_{emergency},$$

其中，原 00:00 计划电量照付；增购部分按 `1.5p`，减购部分另付 `0.5p`；实际供电不足时的紧急购电按 `5p`。

回测采用 10 分钟实际运行粒度，评价期为 2025-02-01 至 2025-12-31；2025 年 1 月用于连续传递储能状态。所有输出写入 `.tmp/q3_baseline_outputs/`。

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import coo_matrix, csr_matrix, hstack

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

ROOT = Path.cwd()
while not (ROOT / 'C题.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

ACTUAL_PATH = ROOT / 'q2' / 'input' / 'q2_actual_load_pv.csv'
PRICE_PATH = ROOT / 'q2' / 'input' / 'q2_fixed_prices.csv'
FORECAST_PATH = ROOT / 'Data_preprocessed' / 'processed' / 'pv_forecast_10min.csv'
OUTPUT_DIR = ROOT / '.tmp' / 'q3_baseline_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'interval_minutes': 10,
    'interval_hours': 1 / 6,
    'charge_efficiency': 0.9,
    'discharge_efficiency': 0.9,
    'storage_min_kwh': 1200.0,
    'storage_max_kwh': 10800.0,
    'initial_storage_kwh': 6000.0,
    'planned_terminal_kwh': 6000.0,
    'max_charge_power_kw': 5000.0,
    'max_discharge_power_kw': 5000.0,
    'emergency_multiplier': 5.0,
    'adjustment_up_multiplier': 1.5,
    'adjustment_down_multiplier': 0.5,
    'history_days': 28,
    'tolerance': 1e-6,
}

UPDATE_BLOCKS = {
    '策略一：06时+12时更新': (6, 12),
    '策略二：仅12时更新': (12,),
}
print('输入文件：', ACTUAL_PATH, PRICE_PATH, FORECAST_PATH)
print('输出目录：', OUTPUT_DIR)

## 1. 数据读取与时间对齐

输入：

- `load_kw`、`pv_kw`：附件 2 的实际负荷和实际光伏功率，功率单位为 kW；
- `price_yuan_per_kwh`：附件 1 的固定日内电价，单位为 元/kWh；
- `pv_forecast_10min.csv`：由附件 3 的四个发布时刻预报整理得到的 10 分钟光伏预报。

运行模型时，功率乘以 `1/6` 转为每个 10 分钟区间的电量。每个发布时刻的预测场景为：当前发布版本的预测值，加上历史同一发布时刻、同一目标区间的光伏预测误差。

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


actual = pd.read_csv(ACTUAL_PATH, parse_dates=['date']).sort_values(['date', 'slot'])
prices_df = pd.read_csv(PRICE_PATH)
forecasts_df = pd.read_csv(
    FORECAST_PATH,
    parse_dates=['issue_time', 'interval_start', 'interval_end', 'anchor_available_at'],
)

require(len(actual) == 365 * 144, '实际数据应为 365 天 × 144 个 10 分钟区间')
require(actual.groupby('date').size().eq(144).all(), '实际数据存在日期缺行')
require(not actual.duplicated(['date', 'slot']).any(), '实际数据存在重复日期-时段')
require(len(prices_df) == 144, '固定电价应为 144 个时段')
require(len(forecasts_df) > 0, '没有读到光伏预报')

dates = pd.DatetimeIndex(sorted(actual['date'].unique()))
load_kw = actual['load_kw'].to_numpy(float).reshape(len(dates), 144)
pv_kw = actual['pv_kw'].to_numpy(float).reshape(len(dates), 144)
prices = prices_df.sort_values('slot')['price_yuan_per_kwh'].to_numpy(float)
require(np.isfinite(load_kw).all() and np.isfinite(pv_kw).all(), '实际功率含非有限值')
require(np.isfinite(prices).all() and (prices > 0).all(), '电价必须为正且有限')

# forecasts[day, issue_index, slot]；issue_index 对应 00/06/12/18 时。
issue_hours = (0, 6, 12, 18)
forecasts = np.full((len(dates), len(issue_hours), 144), np.nan)
date_to_index = {pd.Timestamp(d): i for i, d in enumerate(dates)}
for (issue_time, issue_hour), block in forecasts_df.assign(
    issue_hour=forecasts_df['issue_time'].dt.hour
).groupby(['issue_time', 'issue_hour']):
    day = pd.Timestamp(issue_time).normalize()
    if day not in date_to_index or issue_hour not in issue_hours:
        continue
    k = date_to_index[day]
    j = issue_hours.index(int(issue_hour))
    target_start = day + pd.Timedelta(hours=int(issue_hour))
    target_times = pd.date_range(target_start, day + pd.Timedelta(hours=24), freq='10min', inclusive='left')
    x = block.set_index('interval_start')['pv_forecast_kw_interpolated'].reindex(target_times)
    require(x.notna().all(), f'{issue_time} 预报无法覆盖当天剩余时段')
    forecasts[k, j, int(issue_hour) * 6:] = x.to_numpy(float)

for j, hour in enumerate(issue_hours):
    require(np.isfinite(forecasts[:, j, hour * 6:]).all(), f'{hour:02d}:00 预报没有完整对齐')
    require((forecasts[:, j, hour * 6:] >= 0).all(), f'{hour:02d}:00 光伏预报不应为负')
print(f'读取 {len(dates)} 天，实际数据形状：{load_kw.shape}，预报数据形状：{forecasts.shape}')
display(actual.head())

## 2. 00:00 计划与日内调整模型

00:00 计划对历史误差场景共同优化普通购电 `g`、充电 `c`、放电 `d` 和储能状态 `E`。日内更新时，只允许未来 6 小时的 `g` 改变，其他时段的购电计划锁定为 00:00 计划。

实际执行采用因果控制器：当前区间先用实际光伏和计划购电满足负荷；有余量则充电，有缺口则放电，仍不足的部分记为紧急购电。该执行过程只读取当前区间实际值，因此不会把未来实际值泄漏到决策中。

In [ ]:
def scenario_net_load(load, pv, forecast_cube, day_index, issue_index):
    """构造当前发布时间到 24:00 的历史误差场景，输出 kWh。"""
    start = issue_hours[issue_index] * 6
    first = max(0, day_index - CFG['history_days'])
    current_forecast = forecast_cube[day_index, issue_index, start:]
    history_errors = pv[first:day_index, start:] - forecast_cube[first:day_index, issue_index, start:]
    scenario_pv = np.maximum(current_forecast[None, :] + history_errors, 0.0)
    scenario_load = load[first:day_index, start:]
    return (scenario_load - scenario_pv) * CFG['interval_hours']


def _storage_problem_parts(net_scenarios, p, initial_kwh, fixed_grid=None, lock_from=None):
    """Build the LP/MILP arrays shared by the planning and adjustment problems."""
    a = np.asarray(net_scenarios, float)
    p = np.asarray(p, float)
    s, n = a.shape
    ec, ed = CFG['charge_efficiency'], CFG['discharge_efficiency']
    mc = CFG['max_charge_power_kw'] * CFG['interval_hours']
    md = CFG['max_discharge_power_kw'] * CFG['interval_hours']
    lo, hi = CFG['storage_min_kwh'], CFG['storage_max_kwh']

    # x = [g(n), c(n), d(n), E(n+1), h(s*n), u(n), v(n)]
    h0 = 4 * n + 1
    u0 = h0 + s * n
    v0 = u0 + n
    m = v0 + n
    obj = np.zeros(m)
    obj[h0:u0] = np.tile(CFG['emergency_multiplier'] * p / s, s)

    lower = np.zeros(m)
    upper = np.full(m, np.inf)
    upper[n:2*n] = mc
    upper[2*n:3*n] = md
    lower[3*n:4*n+1] = lo
    upper[3*n:4*n+1] = hi
    lower[3*n] = upper[3*n] = initial_kwh
    lower[4*n] = upper[4*n] = CFG['planned_terminal_kwh']

    t = np.arange(n)
    # Storage dynamics: E[t+1] - E[t] - eta_c*c[t] + d[t]/eta_d = 0.
    eq_rows = np.r_[t, t, t, t]
    eq_cols = np.r_[n+t, 2*n+t, 3*n+t, 3*n+t+1]
    eq_data = np.r_[-ec*np.ones(n), np.ones(n)/ed, -np.ones(n), np.ones(n)]
    storage_eq = coo_matrix((eq_data, (eq_rows, eq_cols)), shape=(n, m)).tocsr()

    scenario_rows = np.arange(s*n)
    scenario_slots = np.tile(t, s)
    # -g - d + c - h <= -net  <=>  g + d - c + h >= net.
    scenario_ub = coo_matrix(
        (
            np.r_[-np.ones(s*n), np.ones(s*n), -np.ones(s*n), -np.ones(s*n)],
            (
                np.tile(scenario_rows, 4),
                np.r_[scenario_slots, n+scenario_slots, 2*n+scenario_slots, h0+scenario_rows],
            ),
        ),
        shape=(s*n, m),
    ).tocsr()
    scenario_rhs = -a.ravel()

    # q - u + v = base, where q is the updated grid purchase plan.
    if fixed_grid is not None:
        base = np.asarray(fixed_grid, float)
        require(len(base) == n, '调整区间计划长度不一致')
        all_eq = storage_eq
        # Append q-u+v=base as a second equality block.
        grid_eq_full = coo_matrix(
            (
                np.r_[np.ones(n), -np.ones(n), np.ones(n)],
                (np.r_[t, t, t], np.r_[t, u0+t, v0+t]),
            ),
            shape=(n, m),
        ).tocsr()
        from scipy.sparse import vstack
        all_eq = vstack([storage_eq, grid_eq_full]).tocsr()
        all_rhs = np.r_[np.zeros(n), base]
    else:
        all_eq = storage_eq
        all_rhs = np.zeros(n)

    if fixed_grid is not None:
        up_coef = CFG['adjustment_up_multiplier'] * p
        down_coef = CFG['adjustment_down_multiplier'] * p
        obj[u0:v0] = up_coef
        obj[v0:] = down_coef
        if lock_from is not None:
            lower[lock_from:n] = base[lock_from:]
            upper[lock_from:n] = base[lock_from:]

    return {
        'a': a, 'p': p, 's': s, 'n': n, 'h0': h0, 'u0': u0, 'v0': v0,
        'm': m, 'obj': obj, 'lower': lower, 'upper': upper,
        'eq': all_eq, 'eq_rhs': all_rhs,
        'scenario_ub': scenario_ub, 'scenario_rhs': scenario_rhs,
        'mc': mc, 'md': md,
    }


def solve_lp_or_milp(parts):
    """Solve the linear relaxation; use MILP only if simultaneous c/d appears."""
    bounds = np.column_stack([parts['lower'], parts['upper']])
    lp = linprog(
        parts['obj'],
        A_ub=parts['scenario_ub'], b_ub=parts['scenario_rhs'],
        A_eq=parts['eq'], b_eq=parts['eq_rhs'], bounds=bounds, method='highs',
    )
    require(lp.success, 'LP 求解失败：' + lp.message)
    x = lp.x
    solver = 'LP'
    n, m = parts['n'], parts['m']
    simultaneous = (x[n:2*n] > CFG['tolerance']) & (x[2*n:3*n] > CFG['tolerance'])
    if simultaneous.any():
        t = np.arange(n)
        mode = coo_matrix(
            (
                np.r_[np.ones(n), -parts['mc']*np.ones(n), np.ones(n), parts['md']*np.ones(n)],
                (
                    np.r_[t, t, n+t, n+t],
                    np.r_[n+t, m+t, 2*n+t, m+t],
                ),
            ),
            shape=(2*n, m+n),
        ).tocsr()
        res = milp(
            np.r_[parts['obj'], np.zeros(n)],
            integrality=np.r_[np.zeros(m), np.ones(n)],
            bounds=Bounds(np.r_[parts['lower'], np.zeros(n)], np.r_[parts['upper'], np.ones(n)]),
            constraints=[
                LinearConstraint(hstack([parts['eq'], csr_matrix((parts['eq'].shape[0], n))]), parts['eq_rhs'], parts['eq_rhs']),
                LinearConstraint(hstack([parts['scenario_ub'], csr_matrix((parts['scenario_ub'].shape[0], n))]), -np.inf, parts['scenario_rhs']),
                LinearConstraint(mode, -np.inf, np.r_[np.zeros(n), np.full(n, parts['md'])]),
            ],
            options={'mip_rel_gap': 1e-8},
        )
        require(res.success, 'MILP 求解失败：' + res.message)
        x = res.x[:m]
        solver = 'MILP'

    residual = max(
        float(np.max(np.abs(parts['eq'] @ x - parts['eq_rhs']))),
        float(np.max(parts['scenario_ub'] @ x - parts['scenario_rhs'])),
        0.0,
    )
    require(residual < 1e-5, f'规划约束残差过大：{residual}')
    require(not (((x[n:2*n] > 1e-6) & (x[2*n:3*n] > 1e-6)).any()), '出现同时充放电')
    return x, solver, residual


def plan_day(net_scenarios, p, initial_kwh):
    parts = _storage_problem_parts(net_scenarios, p, initial_kwh)
    x, solver, residual = solve_lp_or_milp(parts)
    n, h0 = parts['n'], parts['h0']
    return {
        'grid': np.maximum(x[:n], 0),
        'charge': np.maximum(x[n:2*n], 0),
        'discharge': np.maximum(x[2*n:3*n], 0),
        'storage': x[3*n:4*n+1],
        'solver': solver,
        'residual': residual,
    }


def adjust_next_six_hours(net_scenarios, p, initial_kwh, base_grid):
    """Only the first six hours of the remaining horizon may be updated."""
    n = len(base_grid)
    parts = _storage_problem_parts(
        net_scenarios, p, initial_kwh, fixed_grid=base_grid, lock_from=min(36, n),
    )
    x, solver, residual = solve_lp_or_milp(parts)
    return np.maximum(x[:n], 0), solver, residual


def execute_actual(grid, load_kwh, pv_kwh, initial_kwh, battery=True):
    """Causal real-time execution for one block."""
    n = len(grid)
    eta_c, eta_d = CFG['charge_efficiency'], CFG['discharge_efficiency']
    mc = CFG['max_charge_power_kw'] * CFG['interval_hours'] if battery else 0.0
    md = CFG['max_discharge_power_kw'] * CFG['interval_hours'] if battery else 0.0
    e = np.empty(n + 1)
    e[0] = initial_kwh
    charge = np.zeros(n)
    discharge = np.zeros(n)
    emergency = np.zeros(n)
    unused = np.zeros(n)
    for t in range(n):
        residual = grid[t] + pv_kwh[t] - load_kwh[t]
        if residual >= 0:
            charge[t] = min(residual, mc, max(0.0, (CFG['storage_max_kwh'] - e[t]) / eta_c))
            unused[t] = residual - charge[t]
        else:
            discharge[t] = min(-residual, md, max(0.0, (e[t] - CFG['storage_min_kwh']) * eta_d))
            emergency[t] = -residual - discharge[t]
        e[t+1] = e[t] + eta_c * charge[t] - discharge[t] / eta_d
    return {
        'charge': charge, 'discharge': discharge, 'emergency': emergency,
        'unused': unused, 'storage': e,
    }


def adjustment_fee(updated_grid, base_grid, p):
    increase = np.maximum(updated_grid - base_grid, 0.0)
    decrease = np.maximum(base_grid - updated_grid, 0.0)
    return p * (CFG['adjustment_up_multiplier'] * increase + CFG['adjustment_down_multiplier'] * decrease)

In [ ]:
# 费用结算的单元测试：原计划 100、实际更新为 120、p=1 时，调整费为 20*1.5；
# 更新为 80 时，调整费为 20*0.5，且原计划 100 的计划费仍然保留。
p_test = np.ones(2)
base_test = np.array([100.0, 100.0])
updated_test = np.array([120.0, 80.0])
expected_test = np.array([20.0 * 1.5, 20.0 * 0.5])
assert np.allclose(adjustment_fee(updated_test, base_test, p_test), expected_test)
print('结算口径检查通过。')

## 3. 两种策略的全年因果回测

每一天的流程是：

1. 00:00 用当天 00:00 预报和历史误差场景制定全天基础计划；
2. 按策略在 06:00 或 12:00 进行一次未来 6 小时调整；
3. 只用当前区间实际负荷、实际光伏和当前储能状态执行；
4. 将当天 24:00 的储能状态传递到下一天。

1 月只用于预热，统计表仅保留 2 月 1 日至 12 月 31 日。策略一和策略二从相同的初始状态、相同的历史场景开始。

In [ ]:
def run_strategy(strategy_name, update_hours, progress_every=60):
        update_indices = {h: issue_hours.index(h) for h in update_hours}
        state = CFG['initial_storage_kwh']
        daily_rows = []
        detail_frames = []
        decision_rows = []

        for k, day in enumerate(dates):
            day_start_state = state
            # 第一天没有历史完整日期，无法构造场景，按题面初始状态做零计划预热。
            if k == 0:
                base_plan = {'grid': np.zeros(144)}
            else:
                scenarios00 = scenario_net_load(load_kw, pv_kw, forecasts, k, 0)
                base_plan = plan_day(scenarios00, prices, state)
            base_grid = np.asarray(base_plan['grid'], float)
            updated_grid = base_grid.copy()
            blocks = []

            for block_index, start in enumerate((0, 36, 72, 108)):
                end = start + 36
                issue_hour = start // 6
                if issue_hour in update_indices and k > 0:
                    issue_index = update_indices[issue_hour]
                    scenarios = scenario_net_load(load_kw, pv_kw, forecasts, k, issue_index)
                    candidate, solver, residual = adjust_next_six_hours(
                        scenarios, prices[start:], state, base_grid[start:],
                    )
                    updated_grid[start:end] = candidate[:36]
                    decision_rows.append({
                        '策略': strategy_name,
                        'date': day.date(),
                        '更新时间': f'{issue_hour:02d}:00',
                        '修改区间': f'{issue_hour:02d}:00-{issue_hour+6:02d}:00',
                        '调整电量_kWh': float(np.abs(candidate[:36] - base_grid[start:end]).sum()),
                        '求解器': solver,
                        '约束残差': residual,
                    })

                # 逐块执行，下一块使用当前块结束时的真实储能状态。
                load_block = load_kw[k, start:end] * CFG['interval_hours']
                pv_block = pv_kw[k, start:end] * CFG['interval_hours']
                execution = execute_actual(
                    updated_grid[start:end], load_block, pv_block, state, battery=(k > 0),
                )
                state = float(execution['storage'][-1])
                blocks.append(execution)

            charge = np.concatenate([x['charge'] for x in blocks])
            discharge = np.concatenate([x['discharge'] for x in blocks])
            emergency = np.concatenate([x['emergency'] for x in blocks])
            unused = np.concatenate([x['unused'] for x in blocks])
            storage_start = np.concatenate([x['storage'][:-1] for x in blocks])
            storage_end = np.concatenate([x['storage'][1:] for x in blocks])

            plan_fee = prices @ base_grid
            adjustment_components = adjustment_fee(updated_grid, base_grid, prices)
            adjustment_total = float(adjustment_components.sum())
            emergency_fee = float(CFG['emergency_multiplier'] * (prices @ emergency))
            total_fee = float(plan_fee + adjustment_total + emergency_fee)

            daily_rows.append({
                '策略': strategy_name,
                'date': day.date(),
                '计划购电量(kWh)': float(base_grid.sum()),
                '最终计划购电量(kWh)': float(updated_grid.sum()),
                '增购量(kWh)': float(np.maximum(updated_grid - base_grid, 0).sum()),
                '减购量(kWh)': float(np.maximum(base_grid - updated_grid, 0).sum()),
                '紧急购电量(kWh)': float(emergency.sum()),
                '未利用电量(kWh)': float(unused.sum()),
                '计划费用(元)': float(plan_fee),
                '调整费用(元)': adjustment_total,
                '紧急费用(元)': emergency_fee,
                '总费用(元)': total_fee,
                '期初储能(kWh)': float(day_start_state),
                '期末储能(kWh)': float(state),
                '发生紧急购电': bool(emergency.sum() > 1e-6),
            })

            times = pd.date_range(day, periods=144, freq='10min')
            detail_frames.append(pd.DataFrame({
                '策略': strategy_name,
                'date': day.date(),
                'slot': np.arange(1, 145),
                'interval_start': times,
                'interval_end': times + pd.Timedelta(minutes=10),
                'price_yuan_per_kwh': prices,
                'actual_load_kwh': load_kw[k] * CFG['interval_hours'],
                'actual_pv_kwh': pv_kw[k] * CFG['interval_hours'],
                'planned_grid_kwh': base_grid,
                'updated_grid_kwh': updated_grid,
                'increase_kwh': np.maximum(updated_grid - base_grid, 0),
                'decrease_kwh': np.maximum(base_grid - updated_grid, 0),
                'charge_kwh': charge,
                'discharge_kwh': discharge,
                'emergency_kwh': emergency,
                'unused_kwh': unused,
                'storage_start_kwh': storage_start,
                'storage_end_kwh': storage_end,
                'planned_cost_yuan': prices * base_grid,
                'adjustment_cost_yuan': adjustment_components,
                'emergency_cost_yuan': CFG['emergency_multiplier'] * prices * emergency,
                'total_cost_yuan': prices * base_grid + adjustment_components + CFG['emergency_multiplier'] * prices * emergency,
                'evaluation': day.date() >= pd.Timestamp('2025-02-01').date(),
            }))

            if (k + 1) % progress_every == 0 or k == len(dates) - 1:
                print(f'{strategy_name}: {k + 1}/{len(dates)} 天完成')

        return pd.DataFrame(daily_rows), pd.concat(detail_frames, ignore_index=True), pd.DataFrame(decision_rows)


started = time.perf_counter()
results = {}
for strategy_name, update_hours in UPDATE_BLOCKS.items():
    results[strategy_name] = run_strategy(strategy_name, update_hours)
print(f'两种策略回测完成，用时 {time.perf_counter() - started:.1f} 秒。')

In [ ]:
def verify_detail(detail):
    x = detail[detail['evaluation']].copy()
    balance = (
        x['updated_grid_kwh'] + x['actual_pv_kwh'] + x['discharge_kwh'] + x['emergency_kwh']
        - x['actual_load_kwh'] - x['charge_kwh'] - x['unused_kwh']
    )
    storage = x['storage_end_kwh'] - x['storage_start_kwh'] - CFG['charge_efficiency'] * x['charge_kwh'] + x['discharge_kwh'] / CFG['discharge_efficiency']
    checks = {
        '供需平衡最大残差': float(np.abs(balance).max()),
        '储能更新最大残差': float(np.abs(storage).max()),
        '储能下界违约': float(max(0.0, CFG['storage_min_kwh'] - x['storage_end_kwh'].min())),
        '储能上界违约': float(max(0.0, x['storage_end_kwh'].max() - CFG['storage_max_kwh'])),
        '充放电功率约束违约': float(max(0.0, x[['charge_kwh', 'discharge_kwh']].to_numpy().max() - CFG['max_charge_power_kw'] * CFG['interval_hours'])),
        '同时充放电时段数': int(((x['charge_kwh'] > 1e-6) & (x['discharge_kwh'] > 1e-6)).sum()),
        '费用分解最大残差': float(np.abs(x['total_cost_yuan'] - x[['planned_cost_yuan', 'adjustment_cost_yuan', 'emergency_cost_yuan']].sum(axis=1)).max()),
    }
    assert checks['供需平衡最大残差'] < 1e-5
    assert checks['储能更新最大残差'] < 1e-5
    assert checks['储能下界违约'] < 1e-5 and checks['储能上界违约'] < 1e-5
    assert checks['充放电功率约束违约'] < 1e-5
    assert checks['同时充放电时段数'] == 0
    assert checks['费用分解最大残差'] < 1e-5
    return checks


summaries = []
checks = {}
for name, (daily, detail, decisions) in results.items():
    daily_eval = daily[daily['date'] >= pd.Timestamp('2025-02-01').date()].copy()
    row = daily_eval[['计划购电量(kWh)', '最终计划购电量(kWh)', '增购量(kWh)', '减购量(kWh)', '紧急购电量(kWh)', '未利用电量(kWh)', '计划费用(元)', '调整费用(元)', '紧急费用(元)', '总费用(元)']].sum().to_dict()
    row.update({
        '策略': name,
        '发生紧急购电天数': int(daily_eval['发生紧急购电'].sum()),
        '期初储能(kWh)': float(daily_eval['期初储能(kWh)'].iloc[0]),
        '期末储能(kWh)': float(daily_eval['期末储能(kWh)'].iloc[-1]),
        '更新次数': len(decisions[decisions['date'] >= pd.Timestamp('2025-02-01').date()]),
    })
    summaries.append(row)
    checks[name] = verify_detail(detail)

summary = pd.DataFrame(summaries)
summary = summary[['策略', '计划购电量(kWh)', '最终计划购电量(kWh)', '增购量(kWh)', '减购量(kWh)', '紧急购电量(kWh)', '未利用电量(kWh)', '计划费用(元)', '调整费用(元)', '紧急费用(元)', '总费用(元)', '发生紧急购电天数', '更新次数', '期初储能(kWh)', '期末储能(kWh)']]
best = float(summary['总费用(元)'].min())
summary['相对最低费用多花(元)'] = summary['总费用(元)'] - best
summary['相对最低费用多花比例'] = summary['相对最低费用多花(元)'] / best
strategy2_cost = float(summary.loc[summary['策略'].eq('策略二：仅12时更新'), '总费用(元)'].iloc[0])
summary['相对策略二节省(元)'] = strategy2_cost - summary['总费用(元)']
summary['相对策略二节省比例'] = summary['相对策略二节省(元)'] / strategy2_cost

print('物理约束与结算校验：')
display(pd.DataFrame(checks).T)
print('评价期（2025-02-01 至 2025-12-31）费用对比：')
display(summary.style.format({
    c: '{:,.2f}' for c in summary.columns if c.endswith('(kWh)') or c.endswith('(元)') or '多花' in c or '节省' in c
}).format({'相对最低费用多花比例': '{:.2%}', '相对策略二节省比例': '{:.2%}'}))

## 4. 结果图与输出文件

下面输出：

- `summary.csv`：两种策略的评价期费用汇总；
- `daily_results.csv`：逐日费用、紧急购电和储能状态；
- `decisions.csv`：每次 6 小时更新的调整量；
- `detail_*.csv`：逐 10 分钟审计明细；
- `cumulative_cost.png`：累计实际费用曲线；
- `monthly_cost.png`：月度总费用对比。

“实际应用之后的花费”对应 `总费用(元)`，即计划费用、调整费用和实际紧急购电费用三项之和。

In [ ]:
import matplotlib.pyplot as plt

for name, (daily, detail, decisions) in results.items():
    daily.to_csv(OUTPUT_DIR / f'{name}_daily_results.csv', index=False, encoding='utf-8-sig')
    detail.to_csv(OUTPUT_DIR / f'{name}_detail_10min.csv', index=False, encoding='utf-8-sig')
    decisions.to_csv(OUTPUT_DIR / f'{name}_decisions.csv', index=False, encoding='utf-8-sig')
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(checks).T.to_csv(OUTPUT_DIR / 'checks.csv', encoding='utf-8-sig')

plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(11, 5.5))
for name, (daily, _, _) in results.items():
    x = daily[daily['date'] >= pd.Timestamp('2025-02-01').date()].copy()
    x['date'] = pd.to_datetime(x['date'])
    ax.plot(x['date'], x['总费用(元)'].cumsum() / 1e4, lw=2, label=name)
ax.set_title('两种固定更新策略的累计实际费用')
ax.set_xlabel('日期')
ax.set_ylabel('累计费用（万元）')
ax.grid(True, ls='--', alpha=.35)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'cumulative_cost.png', dpi=180)
plt.show()
plt.close(fig)

monthly = []
for name, (daily, _, _) in results.items():
    x = daily[daily['date'] >= pd.Timestamp('2025-02-01').date()].copy()
    x['month'] = pd.to_datetime(x['date']).dt.month
    monthly.append(x.groupby('month', as_index=False)['总费用(元)'].sum().assign(策略=name))
monthly = pd.concat(monthly, ignore_index=True)
monthly.to_csv(OUTPUT_DIR / 'monthly_cost.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(10, 5))
for name, x in monthly.groupby('策略', sort=False):
    ax.plot(x['month'], x['总费用(元)'] / 1e4, marker='o', lw=2, label=name)
ax.set_xticks(range(2, 13))
ax.set_xlabel('月份')
ax.set_ylabel('月度费用（万元）')
ax.set_title('两种固定更新策略的月度实际费用')
ax.grid(True, ls='--', alpha=.35)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'monthly_cost.png', dpi=180)
plt.show()
plt.close(fig)

print('已保存到：', OUTPUT_DIR)
print('summary.csv：', OUTPUT_DIR / 'summary.csv')

## 5. 结论读取方式

重点查看汇总表的 `总费用(元)`：

- 若策略一小于策略二，说明在本次回测中 06:00 的额外一次更新带来的应急购电节省，超过了它产生的调整费用；
- 若策略二小于策略一，则说明 06:00 更新的收益不足以覆盖调整成本；
- 该比较是固定规则的历史回测结果，不是知道未来实际值后的全局最优解。